# Import the necessary libraries and modules

In [1]:
import torch  # PyTorch library for deep learning
import torch.optim as optim  # Optimizers for gradient-based optimization
import torch.nn as nn  # Neural network modules and loss functions
import copy  # Used for creating deep copies of objects
import time  # Provides time-related functionality
import numpy as np  # NumPy library for numerical computations
import pennylane as qml  # PennyLane for quantum machine learning
import matplotlib.pyplot as plt  # Matplotlib for plotting
from torch.utils.data import DataLoader  # DataLoader for handling data batches
dtype = torch.float  # Define the default data type for PyTorch tensors

In [2]:
def Solution1(r, params):
    """
    Computes the exact values of component g00.

    Args:
        r (torch.Tensor): Distance from the gravitational source.
        params (list): List of parameter contains the mass of the gravitational source.

    Returns:
        torch.Tensor: component g00 values.
    """
    M = params[0]
    return -(1 - (2*M)/r)

def Solution2(r, params):
    """
    Computes the exact values of component g11.

    Args:
        r (torch.Tensor): Distance from the gravitational source.
        params (list): List of parameter contains the mass of the gravitational source.

    Returns:
        torch.Tensor: component g11 values.
    """
    M = params[0]
    return (1 - (2*M)/r)**-1

def grad(outputs, inputs):
    """
    Computes the gradient of outputs with respect to inputs.

    Args:
        outputs (torch.Tensor): Output tensor.
        inputs (torch.Tensor): Input tensor.

    Returns:
        Gradient tensor.
    """
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)[0]

def bc_loss(r, a, params):
    """
    Computes boundary condition loss function (boundary condition experssion).

    Args:
        r (torch.Tensor): Distance from the gravitational source.
        a (torch.Tensor): Alpha predicted by neural network.
        params (list): List of parameter contains the mass of the gravitational source.

    Returns:
        torch.Tensor: Mean of Boundary condition loss function.
    """
    M = params[0]
    return torch.mean((-torch.exp(2*a) + (1 - (2*M)/r))**2)

def phy_loss(r, a, b, dra, drb, d2ra):
    """
    Computes the physics-informed loss function (physics experssion).

    Args:
        r (torch.Tensor): Distance from the gravitational source.
        a (torch.Tensor): Alpha predicted by neural network.
        b (torch.Tensor): Beta predicted by neural network.
        dra (torch.Tensor): First derivative of the predicted alpha with respect to r.
        drb (torch.Tensor): First derivative of the predicted beta with respect to r.
        d2ra (torch.Tensor): Second derivative of the predicted alpha with respect to r.
    Returns:
        torch.Tensor: Mean of physics-informed loss function.
    """
    r00 = (torch.exp(2*(a - b))/r)*(dra*(2 + r*dra - r*drb) + r*d2ra)
    r11 = -((dra**2) - (2/r)*drb - dra*drb + d2ra)
    r22 = torch.exp(-2*b)*(r*(drb - dra) - 1) + 1

    R00 = torch.sum(torch.abs(r00))
    R11 = torch.sum(torch.abs(r11))
    R22 = torch.sum(torch.abs(r22))

    return R00, R11, R22

def train_PINN(model, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model=[False, "model"]):
    """
    Trains a physics-informed neural network (PINN) model.

    Args:
        model: The neural network model.
        batches: Number of points in the domain.
        batch_size: Number of points in each batch.
        domain: Defined domain in the interval [100, 300].
        params (list): List of parameter contains the mass of the gravitational source.
        epochs: Number of epochs.
        num_itr: Number of iteration.
        loss_weights (list): List containing weights of overall loss function.
        save_model: Optional flag for saving the model.

    Returns:
        Trained model, Loss function and its components.
    """
    torch.manual_seed(seed)
    # Initialize the best trained neural network model (bNN) as None
    bNN = None
    # Initialize total loss (tot_l) as None
    tot_l = None
    
    # Initialize empty lists for different loss histories
    phy_loss1_history = [] # R00
    phy_loss2_history = [] # R11
    phy_loss3_history = [] # R22
    bc_loss_history = [] # Boundary condition loss functoin
    overall_loss_history = [] # Overall loss function history per epoch
    overall_wloss_history = [] # Overall weighted loss function history per epoch
    wloss_history = [] # Overall weighted loss function per iteration
    
    # Initialize an upper limit for loss (l_lim) with a large value
    l_lim = 1e+20

    # Generate evenly spaced points within the specified domain
    points = torch.linspace(domain[0], domain[1], batches).view(-1, 1)
    # Enable gradient tracking for these points (useful for optimization)
    points.requires_grad = True
    # Create a data loader for the points with the specified batch size
    loader = DataLoader(points, batch_size, shuffle=True)

    # Record the start time before training begins
    time_i = time.time()

    # Training loop over epochs
    for epoch in range(epochs):
        # Iterate through data loader batches
        for r in loader:
            # Iterate over each batch
            for i in range(num_itr):
                # Pass the input data (feature) through the neural network and obtain alpha and beta
                alpha, beta = model(r)

                # Compute physics-informed and boundary condition losses, then combine them
                dralpha = grad(alpha, r) # Compute first derivative of alpha with respect to r
                drbeta = grad(beta, r) # Compute first derivative of beta with respect to r
                d2ralpha = grad(dralpha, r) # Compute second derivative of alpha with respect to r
                phy_l1, phy_l2, phy_l3 = phy_loss(r, alpha, beta, dralpha, drbeta, d2ralpha)
                bc_l = bc_loss(r, alpha, params)
                tot_wl = loss_weights[0]*phy_l1 + loss_weights[1]*phy_l2 + loss_weights[2]*phy_l3 + loss_weights[3]*bc_l
                
                # Update model parameters: zero gradients, compute backward pass, and perform optimization step
                optimizer.zero_grad()
                tot_wl.backward(retain_graph=True)
                optimizer.step()
                
                # Record overall weighted loss function for every iteratoin
                wloss_history.append(tot_wl.item())
                
        
        # Compute losses and various expressions to record them
        alpha, beta = model(points)
        dralpha = grad(alpha, points)
        drbeta = grad(beta, points)
        d2ralpha = grad(dralpha, points)
        phy_l1, phy_l2, phy_l3 = phy_loss(points, alpha, beta, dralpha, drbeta, d2ralpha)
        bc_l = bc_loss(points, alpha, params)
        tot_wl = loss_weights[0]*phy_l1 + loss_weights[1]*phy_l2 + loss_weights[2]*phy_l3 + loss_weights[3]*bc_l
        tot_l = phy_l1 + phy_l2 + phy_l3 + bc_l

        # Append each losses and various expressions to the respective lists
        phy_loss1_history.append(phy_l1.item())
        phy_loss2_history.append(phy_l2.item())
        phy_loss3_history.append(phy_l3.item())
        bc_loss_history.append(bc_l.item())
        overall_loss_history.append(tot_l.item())
        overall_wloss_history.append(tot_wl.item())
        
        # Print the loss value at specific epoch during training
        print(f'epoch {epoch+1}/{epochs}, loss = {tot_wl}')

        # Update the best neural network if the total loss is lower than the current limit
        if  tot_wl < l_lim:
            bNN =  copy.deepcopy(model)
            l_lim = tot_wl

    # Collect different loss histories for analysis
    loss_histories = [overall_wloss_history, overall_loss_history, phy_loss1_history, phy_loss2_history, phy_loss3_history, bc_loss_history, wloss_history]

    # Find the minimum overall loss value, its epoch, and print the result
    min_wloss = min(overall_wloss_history)
    min_index_wloss = overall_wloss_history.index(min_wloss)
    print(f"The minimum overall weighted loss function value occurs at epoch {min_index_wloss+1}: {overall_wloss_history[min_index_wloss]}")

    # Measure the runtime of the training process
    time_f = time.time()
    runtime = time_f - time_i
    print(f"train took {runtime} s")
    
    # Save model state dictionaries and loss histories if requested
    if save_model[0] == True:
        torch.save(bNN.state_dict(), save_model[1]+".pth")
        torch.save(loss_histories, "loss_"+save_model[1]+".pt")

    return bNN, loss_histories

# Classical Neural Network

In [3]:
def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        m.bias.data.fill_(0.01)

class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with six linear layer followed by a LogSigmoid activation.

    Notes:
        - The classical neural network architecture consists of six hidden layers.
        - The input layer has 'input_size' neurons.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(CNeuralNet, self).__init__()

        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, output_size),
        )

        # self.fnn.apply(init_weights)

    def forward(self, x):
        out = self.fnn(x)
        a = out[:, 0].view(-1, 1)
        b = out[:, 1].view(-1, 1)
        return a, b

In [ ]:
for h_size in [10, 30, 50]:
    # Set the random seed for reproducibility
    seed = 14
    torch.manual_seed(seed)
    # Define neural network with input, hidden, and output sizes
    input_size , hidden_size, output_size = 1, h_size, 2
    model_classic = CNeuralNet(input_size, hidden_size, output_size)
    
    # Set optimization parameters
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.0005
    optimizer = optim.Adam(model_classic.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_classic)
    # Set the necessary parameters for training and test
    M = 15
    params = [M]
    num_itr, epochs, = 50, 20
    batches, batch_size =  128, 32
    domain, loss_weights = [100, 300], [10, 10, 0.001, 10]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"Classical model_{seed}_{h_size}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_classic, loss_hist_classic = train_PINN(model_classic, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)

# Quantum Neural Network

In [ ]:
# Set the number of qubits to 3
n_qubits = 2
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (list): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

@qml.qnode(dev)
def quantum_net(input_features, n_qubits, ansatz_depth, feature_weights, ansatz_weights, out_weights):
    """
    Quantum circuit

    Args:
        input_features: Input features.
        n_qubits (int): Number of qubits in the circuit.
        ansatz_depth (int): Depth of the ansatz (number of layers).
        feature_weights (torch.Tensor): List of parameters.
        ansatz_weights (torch.Tensor): List of parameters.
        out_weights (torch.Tensor): List of parameters.

    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0 and 1 as g00 and g11.
    """
    # apply feature map layers
    temp1 = feature_weights[0]*(input_features[0]**feature_weights[1])
    temp2 = feature_weights[2]*(input_features[0]**feature_weights[3])
    qml.RY(temp1, wires=0)
    qml.RY(temp2, wires=1)
    qml.CNOT(wires=[0,1])

    # apply ansatz layers
    for k in range(ansatz_depth):
        RY_layer(ansatz_weights[k])
        qml.CNOT(wires=[0,1])

    # Compute expectation values for Pauli-Z operators
    exp_vals = [qml.expval(out_weights[0]*qml.PauliZ(0)), qml.expval(out_weights[1]*qml.PauliZ(1))]
    return tuple(exp_vals)


class QNeuralNet(nn.Module):
    """
    quantum neural network class.

    Args:
        n_qubits (int): Number of qubits in the circuit.
        feature_depth (int): Depth of the feature map (number of layers).
        ansatz_depth (int): Depth of the ansatz (number of layers).

    Notes:
        - The quantum neural network architecture consists of two qubits.
        - The feature map has 'feature_depth' layers.
        - The ansatz has 'ansatz_depgh' layers.
    """
    def __init__(self, n_qubits, feature_depth, ansatz_depth):
        super(QNeuralNet, self).__init__()
        
        # Set the number qubits, number of ansatz layer and feature map layers
        self.n_qubits = n_qubits
        self.ansatz_depth = ansatz_depth
        self.feature_depth = feature_depth
        
        # Initialize parameters of quantum circuit
        self.feature_weights = nn.Parameter(0.1*torch.randn((2*n_qubits*feature_depth)).view(-1, 1))
        self.ansatz_weights = nn.Parameter(torch.randn(ansatz_depth*n_qubits).view(ansatz_depth, n_qubits))
        self.out_weights = nn.Parameter(torch.FloatTensor(2).uniform_(-1, 1))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Scale features by dividing by 100
        x = x/100
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, 2)

        for elem in x:
            # Apply the quantum circuit (quantum_net) to each input element
            q_out_elem = torch.hstack(tuple(quantum_net(elem, self.n_qubits, self.ansatz_depth, self.feature_weights,self.ansatz_weights, self.out_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem)) # Concatenate quantum outputs

        # Compute the ansatz solution by applying the arcsine function to the neural network outputs
        out1 = torch.arcsin(q_out[:, 0]).view(-1, 1)
        out2 = torch.arcsin(q_out[:, 1]).view(-1, 1)
        
        return out1, out2

In [ ]:
for depth in [1, 2, 3]:
    # Set the random seed for reproducibility
    seed = 14
    torch.manual_seed(seed)
    # Define quantum neural network by number of qubits, feature layers and ansatz layers
    feature_depth, ansatz_depth = 1, depth
    model_quantum = QNeuralNet(n_qubits, feature_depth, ansatz_depth)
    
    # Set optimization parameters
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.025
    optimizer = optim.Adam(model_quantum.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_quantum)
    # Set the necessary parameters for training and test
    M = 15
    params = [M]
    num_itr, epochs, = 50, 20
    batches, batch_size =  128, 32
    domain, loss_weights = [100, 300], [10, 10, 0.001, 10]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"quantum model_{seed}_{depth}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_classic, loss_hist_quantum = train_PINN(model_quantum, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)

# Hybrid Quantum Neural Network

In [ ]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (list): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

@qml.qnode(dev)
def hquantum_net(input_features, n_qubits, q_depth, q_weights):
    """
    Quantum circuit as a part of quantum hybrid neural network

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        q_weights (torch.Tensor): List of parameters.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2.
    """
    RY_layer(input_features[0])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[1])
    RY_layer(q_weights[0])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[2])
    RY_layer(q_weights[1])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[3])
    RY_layer(q_weights[2])
    entangling_layer(n_qubits)

    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(0, n_qubits)]
    return tuple(exp_vals)

class HQNeuralNet(nn.Module):
    """
    quantum hybrid neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with a linear layer followed by a Tanh activation.

    Notes:
        - The first classical neural network architecture consists of one hidden layer.
        - The input layer has 'input_size' neurons.
        - The hidden layers has 'hidden_size' neurons with a Tanh activation.
        - The quantum neural network architecture consists of 'n_qubits' and 'q_depth ' layers.
        - The second classical neural network architecture consists of one linear layer.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, n_qubits, q_depth, output_size):
        super().__init__()
        
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, (q_depth+1)*n_qubits),
        )
        # self.fnn.apply(init_weights)

        #Define the feedforward (n_qubits, output_size) layer as a output layer
        self.output_layer = nn.Linear(n_qubits, output_size)
        # self.output_layer.apply(init_weights)

        # Set the number qubits, number of quantum layers
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        
        # Initialize parameters of quantum circuit
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Pass the data through the feedforward neural network (self.fnn)
        out = self.fnn(x)
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, self.n_qubits)

        for elem in out:
            # Apply the quantum circuit (hquantum_net) to each output element
            elem = elem.view((self.q_depth+1), self.n_qubits)
            q_out_elem = torch.hstack(tuple(hquantum_net(elem, self.n_qubits, self.q_depth, self.q_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))

        # Pass the output of quantum circuit through the feedforward linear layer
        out = self.output_layer(q_out)
        a = out[:, 0].view(-1, 1)
        b = out[:, 1].view(-1, 1)
        return a, b

In [ ]:
for h_size in [10, 30, 50]:
    # Set the random seed for reproducibility
    seed = 14
    torch.manual_seed(seed)
    # Define input, hidden, and output sizes and specify the domain (range) for input data
    input_size, hidden_size, q_depth, output_size = 1, h_size, 3, 2
    model_hybrid = HQNeuralNet(input_size, hidden_size, n_qubits, q_depth, output_size)
    
    # Create a hybrid quantum neuralnet class and print the model architecture
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.0005
    optimizer = optim.Adam(model_hybrid.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_hybrid)
    # Set the necessary parameters for training and test
    M = 15
    params = [M]
    num_itr, epochs, = 50, 20
    batches, batch_size =  128, 32
    domain, loss_weights = [100, 300], [10, 10, 0.001, 10]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"hybrid model_{seed}_{h_size}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_hybrid, loss_hist_hybrid = train_PINN(model_hybrid, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)